# 08. Motor + chassis + battery: a multi-point Pareto front

A small electric vehicle is co-designed from three subsystems:

- a **motor** picked from a discrete catalog (each entry has its own `(torque, mass, cost)` tuple),
- a **chassis** whose mass and cost scale with the load it supports,
- a **battery** sized to the mission energy.

The subsystems are coupled cyclically: the chassis must support the motor and battery, the motor's torque is sized by the total moving mass (which includes the chassis), and so on. The `System` builder closes the loop.

Because the motor catalog has Pareto-incomparable entries, the system-level Pareto front has multiple points: real engineering tradeoffs surface automatically.

This notebook also uses the operator-overloaded constraint syntax introduced in notebook 07.


## Imports

In [1]:
from codesign import (
    CatalogDP, CatalogEntry, Module, NamedProduct, Reals,
    System, minimize_cost, solve,
)

## Subsystem 1: a motor catalog

The catalog has Pareto-incomparable entries (lighter and more expensive vs. heavier and cheaper). `CatalogDP` is kept as a plain function constructor: multi-valued antichains don't fit the `Module` declarative pattern as cleanly.


In [2]:
motor = CatalogDP(
    F=NamedProduct({"torque": Reals(unit="N*m")}),
    R=NamedProduct({"mass": Reals(unit="kg"), "cost": Reals(unit="USD")}),
    catalog=[
        CatalogEntry(name="Tiny",          provides={"torque": 2.0},  costs={"mass": 0.20, "cost": 30.0}),
        CatalogEntry(name="Light-Premium", provides={"torque": 8.0},  costs={"mass": 0.50, "cost": 200.0}),
        CatalogEntry(name="Mid-Standard",  provides={"torque": 8.0},  costs={"mass": 0.80, "cost": 120.0}),
        CatalogEntry(name="Heavy-Budget",  provides={"torque": 20.0}, costs={"mass": 1.50, "cost": 90.0}),
        CatalogEntry(name="Light-Pro",     provides={"torque": 20.0}, costs={"mass": 0.90, "cost": 350.0}),
        CatalogEntry(name="XL-Budget",     provides={"torque": 80.0}, costs={"mass": 3.50, "cost": 180.0}),
        CatalogEntry(name="XL-Pro",        provides={"torque": 80.0}, costs={"mass": 2.20, "cost": 700.0}),
    ],
    name="motor",
)
motor

DP(motor: torque:R+[N*m] -> A[mass:R+[kg]×cost:R+[USD]])

## Subsystems 2 and 3: chassis and battery as Module classes

In [3]:
class Chassis(Module):
    F = {"load": Reals(unit="kg")}
    R = {"mass": Reals(unit="kg"), "cost": Reals(unit="USD")}

    def h(self, f):
        return {
            "mass": 0.6  * f["load"],
            "cost": 20.0 * f["load"],
        }


class Battery(Module):
    F = {"energy": Reals(unit="J")}
    R = {"mass":   Reals(unit="kg"), "cost": Reals(unit="USD")}

    def h(self, f):
        return {
            "mass": f["energy"] / 1.8e6,
            "cost": 0.05 * f["energy"] / 3.6e3,  # $0.05 per Wh
        }

## Wire it up with `>=`

The chassis must support payload plus motor plus battery; the motor's torque demand depends on the total moving mass; the battery's energy demand is set externally. Total mass and total cost aggregate from every subsystem.


In [4]:
G = 9.81
TORQUE_PER_KG = 0.25

sys = System("vehicle")

payload        = sys.provides("payload",        unit="kg")
mission_energy = sys.provides("mission_energy", unit="J")
total_mass     = sys.requires("total_mass",     unit="kg")
total_cost     = sys.requires("total_cost",     unit="USD")

m = sys.add("motor",   motor)
c = sys.add("chassis", Chassis())
b = sys.add("battery", Battery())

c.load   >= payload + m.mass + b.mass
m.torque >= TORQUE_PER_KG * G * (payload + c.mass + b.mass)
b.energy >= mission_energy

total_mass >= payload + m.mass + c.mass + b.mass
total_cost >= m.cost + c.cost + b.cost

vehicle = sys.build()
print(sys)

System('vehicle'):
  provides:
    payload: R+[kg]
    mission_energy: R+[J]
  requires:
    total_mass: R+[kg]
    total_cost: R+[USD]
  subsystems:
    motor: (torque) -> (mass, cost)
    chassis: (load) -> (mass, cost)
    battery: (energy) -> (mass, cost)
  constraints:
    chassis.load >= ((payload + motor.mass) + battery.mass)
    motor.torque >= (2.4525 * ((payload + chassis.mass) + battery.mass))
    battery.energy >= mission_energy
    total_mass >= (((payload + motor.mass) + chassis.mass) + battery.mass)
    total_cost >= ((motor.cost + chassis.cost) + battery.cost)


## Solve for three mission profiles

In [5]:
cases = [
    ("Small parcel", dict(payload=2.0,  mission_energy=2.0e5)),
    ("Medium load",  dict(payload=10.0, mission_energy=1.0e6)),
    ("Heavy + long", dict(payload=20.0, mission_energy=5.0e6)),
]
for label, f in cases:
    result = solve(vehicle, f, max_iter=200)
    f_str = ", ".join(f"{k}={v}" for k, v in f.items())
    print(f"{label}: {f_str}")
    print(f"   iters={result.iterations}, feasible={result.feasible}")
    if not result.feasible:
        print()
        continue
    print(f"   Pareto front ({len(result.antichain.points)} points):")
    for p in result.antichain.points:
        print(f"      total_mass={p['total_mass']:6.2f} kg,  "
              f"total_cost=${p['total_cost']:7.2f}")
    cheapest = minimize_cost(result, cost_fn=lambda r: r["total_cost"])
    if cheapest is not None:
        print(f"   cheapest: total_mass={cheapest['total_mass']:.2f} kg, "
              f"total_cost=${cheapest['total_cost']:.2f}")
    print()

Small parcel: payload=2.0, mission_energy=200000.0
   iters=4, feasible=True
   Pareto front (2 points):
      total_mass=  4.82 kg,  total_cost=$ 413.00
      total_mass=  5.78 kg,  total_cost=$ 165.00
   cheapest: total_mass=5.78 kg, total_cost=$165.00

Medium load: payload=10.0, mission_energy=1000000.0
   iters=3, feasible=True
   Pareto front (2 points):
      total_mass= 22.49 kg,  total_cost=$ 475.00
      total_mass= 20.41 kg,  total_cost=$ 969.00
   cheapest: total_mass=22.49 kg, total_cost=$475.00

Heavy + long: payload=20.0, mission_energy=5000000.0
   iters=2, feasible=False



## Reading the Pareto fronts

For Small parcel and Medium load, two motor choices remain feasible after the chassis grows to support everything, and they trade total mass against total cost: one is lighter but more expensive, the other heavier but cheaper. Neither dominates, so both appear in the system Pareto front.

For Heavy + long, even the biggest motor in the catalog can't supply enough torque once the chassis (which scales with battery mass) grows large enough. The iteration drives the loop axis to `⊤` and the result is `feasible=False`, automatically and without any special handling in the example code.

This is the payoff of modularity: each subsystem has a clean local definition, and the global Pareto structure is discovered by the solver from the constraint graph alone.
